# Offline inference

Attach competition data, the [wheelhouse](https://www.kaggle.com/datasets/denysmykhailov/birdclef-2026-wheelhouse),
and a model Dataset or a training notebook's saved Output. The [model Dataset](https://www.kaggle.com/datasets/denysmykhailov/birdclef-2026-models) has two example runs.
Choose CPU (Accelerator: None) and Internet OFF.

A run needs config, classes and checkpoints. Use the best checkpoint from
`fit.json`, or `last.ckpt` when no best is recorded. For local models, export
`build/birdclef-models.zip` and upload it as a Dataset.
No training, weight downloads or W&B run starts here.

In [ ]:
from pathlib import Path

wheel, = [
    path for path in Path("/kaggle/input").rglob("birdclef*.whl")
    if (path.parent / "requirements.txt").is_file()
]
bundle = wheel.parent
%pip install --no-index --find-links "{bundle}" -r "{bundle}/requirements.txt" -q
%pip install --no-index --force-reinstall --no-deps "{wheel}" -q

In [ ]:
import os

CPU_THREADS = 4
for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[name] = str(CPU_THREADS)

available_runs = sorted(path.parent for path in Path("/kaggle/input").rglob("model-config.json"))
for run in available_runs:
    print(run)
assert available_runs, "Attach a model Dataset or the saved Output of a training notebook."

## Choose models

`RUN_DIRS` starts with the first printed run. Use `[available_runs[1]]` for the
second, or `available_runs[:2]` to average the first two. Explicit paths also work.
`WEIGHTS=None` gives equal weights; a list such as `[0.7, 0.3]` follows model order.
Do not select smoke runs. Save one version per model for separate submissions.

In [ ]:
RUN_DIRS = [available_runs[0]]
WEIGHTS = None
BATCH_SIZE = 32
NUM_WORKERS = 0

COMPETITION = next(
    path.parent for path in Path("/kaggle/input").rglob("sample_submission.csv")
    if path.parent.name == "birdclef-2026"
)
OUTPUT = Path("/kaggle/working/submission.csv")

In [ ]:
import torch
from birdclef.submission import run_inference

torch.set_num_threads(CPU_THREADS)
run_inference(
    RUN_DIRS, COMPETITION, OUTPUT,
    weights=WEIGHTS, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
)

## Output

With no public test audio, Run All checks one real training minute and saves
`smoke-predictions.csv`. It copies sample rows to `submission.csv`.
During scoring, it predicts all actual test files and writes real scores.
IDs come from filenames and 5-second end times. The short public check does not
measure full test runtime or model quality.

In [ ]:
import pandas as pd

submission = pd.read_csv(OUTPUT)
print(f"{OUTPUT}: {submission.shape[0]} rows, {submission.shape[1] - 1} classes")
submission.head()